In [1]:
%pip install numpy


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Environment Setup and Cluster Configuration
First, we need to initialize our PySpark environment specifically tailored for the Aion cluster. To handle the computational weight of matrix factorization, I am allocating 20GB to the executors and 10GB to the driver, while utilizing the highly optimized `KryoSerializer`. This ensures we don't hit memory limits during the intensive shuffle operations later on.

In [2]:
import findspark

spark_path='/opt/apps/easybuild/systems/aion/rhel810-20250803/2023b/epyc/software/Spark/3.5.4-foss-2023b-Java-17'
findspark.init(spark_path)

from pyspark.sql import SparkSession
import os

# Double-check Spark can see the Java home too, just in case
print(f"Spark Home: {os.environ.get('SPARK_HOME')}")

spark = SparkSession.builder \
    .appName("AudioScrobbler_Recommender_HPC") \
    .config("spark.executor.memory", "20g") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.cores", "4") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark Session Created Successfully!")

Spark Home: /opt/apps/easybuild/systems/aion/rhel810-20250803/2023b/epyc/software/Spark/3.5.4-foss-2023b-Java-17


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 13:57:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully!


## 2. Data Ingestion and Alias Resolution
Here, we load the raw AudioScrobbler dataset. A major data-quality issue in this dataset is that the same artist often appears under multiple IDs (aliases/misspellings). To clean this, I load the `artist_alias.txt` file, map it to a dictionary, and broadcast it to all worker nodes. When building our final `Rating` objects, this broadcast variable ensures every interaction points to the canonical artist ID without requiring an expensive distributed join.

In [3]:
from pyspark.mllib.recommendation import ALS, Rating

# Load raw interaction data
PATH_TO_DATA = "./audioscrobbler"
raw_user_artist_data = sc.textFile(f"{PATH_TO_DATA}/user_artist_data.txt")
raw_artist_alias = sc.textFile(f"{PATH_TO_DATA}/artist_alias.txt")

# Load and process artist aliases into a dictionary
def parse_alias(line):
    tokens = line.split('\t')
    if len(tokens) < 2 or not tokens[0]:
        return None
    return (int(tokens[0]), int(tokens[1]))

artist_alias = raw_artist_alias.map(parse_alias).filter(lambda x: x is not None).collectAsMap()

# Broadcast the alias dictionary to all nodes
alias_broadcast = sc.broadcast(artist_alias)

# Map interactions to consistent artist IDs and create Rating objects
def build_ratings(line):
    user, artist, count = map(int, line.split())
    # Redirect alias if it exists, otherwise keep original
    final_artist = alias_broadcast.value.get(artist, artist)
    return Rating(user, final_artist, count)

trainData = raw_user_artist_data.map(build_ratings)

## 3. Filtering for Active Users (Step A.i)
Following the exercise requirements, our model should only consider "active" users to ensure sufficient data density for matrix factorization. I count the total unique artists per user and filter for those with $\ge 100$ artists. Finally, I join this filtered list back to the main dataset to isolate our `trainData100` RDD. Caching this result is critical to prevent Spark from recomputing this lineage repeatedly.

In [4]:
# 1. Count artists per user
user_counts = trainData.map(lambda r: (r.user, 1)).reduceByKey(lambda a, b: a + b)

# 2. Filter for users with >= 100 artists
# The assignment notes this should result in 72,748 users
active_user_ids = user_counts.filter(lambda x: x[1] >= 100).map(lambda x: x[0])

# 3. Join back to get the original Rating objects for only these users
# We use a join here to ensure trainData100 only has 'active' ratings
trainData100 = trainData.map(lambda r: (r.user, r)) \
                       .join(active_user_ids.map(lambda u: (u, None))) \
                       .map(lambda x: x[1][0]) \
                       .cache() # CRITICAL: Don't let Spark re-run this expensive join!
print(f"Total active users {active_user_ids.count()}")
print(f"Total records in trainData100: {trainData100.count()}")

Total active users 72748


Total records in trainData100: 21300914


## 4. The Intra-User 80/20 Split (Step A.ii)
A standard global `randomSplit` would ruin the evaluation by entirely removing some users from the training set. The requirement dictates a strict per-user split. By grouping the ratings by `user_id` and applying a Python shuffle mapping, I ensure that exactly 80% of *each specific user's* listening history goes to `trainData_final` and the remaining 20% is held out in `testData_final` for evaluation.

In [5]:
import random

def split_per_user(user_ratings_tuple):
    user_id, ratings_iter = user_ratings_tuple
    ratings = list(ratings_iter)
    random.seed(42) # Keeps results consistent
    random.shuffle(ratings)
    
    # Calculate the 80% mark for THIS specific user
    split_point = int(len(ratings) * 0.8)
    
    return (ratings[:split_point], ratings[split_point:])

# 1. Group by user ID
grouped_by_user = trainData100.map(lambda r: (r.user, r)).groupByKey()

# 2. Apply the 80/20 split to each user's list
# This creates an RDD of tuples: ( [80% list], [20% list] )
splits = grouped_by_user.map(split_per_user).cache()

# 3. Flatten them back into separate RDDs of Rating objects
trainData_final = splits.flatMap(lambda x: x[0]).cache()
testData_final = splits.flatMap(lambda x: x[1]).cache()

print(f"Training items: {trainData_final.count()}")
print(f"Testing items: {testData_final.count()}")

Training items: 17011713


Testing items: 4289201


## 5. Checkpointing & Baseline Model Initialization
Before running iterative algorithms like ALS, we must define a checkpoint directory on the Aion scratch space. Without this, Spark's lineage graph grows infinitely during iterations, resulting in `StackOverflow` errors and cluster deadlocks. Once secured, we train a baseline ALS model using default arbitrary parameters (`rank=10`, `lambda=0.01`) to establish our initial benchmark.

In [6]:

sc.setCheckpointDir("/scratch/users/eleiva/spark_checkpoints") 

model = ALS.trainImplicit(trainData_final, rank=10, iterations=10, lambda_=0.01, alpha=1.0)

26/05/03 13:59:50 WARN BlockManager: Task 158 already completed, not releasing lock for rdd_24_0
26/05/03 13:59:55 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/03 13:59:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


## 6. Distributed Evaluation & Baseline Comparison (Step A.iii)
To evaluate the model, we need to generate Top-50 recommendations for all test users and compare the results against their actual hidden artists. Instead of bottlenecking the driver node with a sequential `for` loop, I leverage `recommendProductsForUsers(50)` to trigger a distributed matrix multiplication. Simultaneously, we compute a "Most Popular" baseline model to see if our ALS algorithm actually learns latent features or just recommends famous bands.

In [7]:
import numpy as np

# 1. Calculate Global Popularity ONCE
item_popularity = trainData_final.map(lambda r: (r.product, 1)) \
                                 .reduceByKey(lambda a, b: a + b) \
                                 .collectAsMap()
top_50_popular = sorted(item_popularity.items(), key=lambda x: x[1], reverse=True)[:50]
pop_broadcast = sc.broadcast(top_50_popular)

# 2. Group test data by user: RDD of (user, set(actual_artists))
user_test_artists = testData_final.map(lambda r: (r.user, r.product)) \
                                  .groupByKey() \
                                  .mapValues(set)

# 3. Generate recommendations for ALL users in ONE distributed job
# Returns RDD of (user, (Rating1, Rating2, ...))
all_recommendations = model.recommendProductsForUsers(50)

# 4. Join predictions with actuals and compute AUC per user on the Worker Nodes
def compute_worker_auc(row):
    user, (recs, actual_artists) = row
    
    # --- ALS Model AUC ---
    model_pairs = [(float(r.rating), 1.0 if r.product in actual_artists else 0.0) for r in recs]
    sorted_model = sorted(model_pairs, key=lambda x: x[0], reverse=True)
    model_labels = [x[1] for x in sorted_model]
    
    # --- Popularity Baseline AUC ---
    pop_pairs = [(float(score), 1.0 if artist in actual_artists else 0.0) for artist, score in pop_broadcast.value]
    sorted_pop = sorted(pop_pairs, key=lambda x: x[0], reverse=True)
    pop_labels = [x[1] for x in sorted_pop]
    
    def calc_auc(labels):
        n_pos = sum(labels)
        n_neg = len(labels) - n_pos
        if n_pos == 0 or n_neg == 0: return 0.5
        tp, fp, auc = 0, 0, 0
        for label in labels:
            if label == 1:
                tp += 1
            else:
                auc += tp
                fp += 1
        return auc / (n_pos * n_neg)

    return (calc_auc(model_labels), calc_auc(pop_labels))

# Perform the join and map
eval_rdd = all_recommendations.join(user_test_artists).map(compute_worker_auc)

# Collect the final averages back to the driver
total_users = eval_rdd.count()
sum_aucs = eval_rdd.reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))

print(f"Average ALS AUC: {sum_aucs[0] / total_users:.4f}")
print(f"Average Popularity AUC: {sum_aucs[1] / total_users:.4f}")

Average ALS AUC: 0.5593
Average Popularity AUC: 0.5286


## 7. Hyperparameter Tuning via 5-Fold Cross-Validation (Step B)
To find the optimal model, we execute a manual triple `for` loop over our parameter grids for `rank`, `lambda`, and `alpha`. For robust evaluation, I implemented a 5-fold cross-validation. 

*Note for Aion Execution:* Running 135 model iterations is extremely heavy. To prevent cluster timeouts and memory leaks, I explicitly sever the RDD lineage using `.checkpoint()` inside the fold union, and utilize a distributed worker-side function to calculate the AUC for each fold rapidly.

In [ ]:
# Setup ranges
ranks = [10, 25, 50]
lambdas = [1.0, 0.1, 0.01]
alphas = [1.0, 10.0, 100.0]

print("Splitting training data into 5 folds...")
folds = trainData_final.randomSplit([0.2, 0.2, 0.2, 0.2, 0.2], seed=42)
for fold in folds:
    fold.cache()

best_auc = 0.0
best_params = {}

print("Starting Scalable Triple-Loop Grid Search...")

for r in ranks:
    for l in lambdas:
        for a in alphas:
            cv_aucs = []
            for i in range(5):
                cv_train_list = [folds[j] for j in range(5) if i != j]
                cv_train = sc.union(cv_train_list)
                
                # CRITICAL: Sever lineage to prevent StackOverflows in iterative loops
                cv_train.checkpoint() 
                cv_val = folds[i]
                
                # 1. Train Model
                model = ALS.trainImplicit(cv_train, rank=r, iterations=10, lambda_=l, alpha=a)
                
                # 2. Fast Distributed Evaluation
                val_actuals = cv_val.map(lambda row: (row.user, row.product)).groupByKey().mapValues(set)
                val_recs = model.recommendProductsForUsers(50)
                
                # Define evaluation for worker
                def eval_fold(row):
                    user, (recs, actual_artists) = row
                    pairs = [(float(rec.rating), 1.0 if rec.product in actual_artists else 0.0) for rec in recs]
                    pairs.sort(key=lambda x: x[0], reverse=True)
                    labels = [x[1] for x in pairs]
                    n_pos = sum(labels)
                    n_neg = len(labels) - n_pos
                    if n_pos == 0 or n_neg == 0: return 0.5
                    tp, auc = 0, 0
                    for label in labels:
                        if label == 1: tp += 1
                        else: auc += tp
                    return auc / (n_pos * n_neg)

                fold_auc_rdd = val_recs.join(val_actuals).map(eval_fold)
                avg_fold_auc = fold_auc_rdd.mean()
                cv_aucs.append(avg_fold_auc)
            
            avg_cv_auc = np.mean(cv_aucs)
            print(f"Tested: Rank={r}, Lambda={l}, Alpha={a} | CV Avg AUC: {avg_cv_auc:.4f}")
            
            if avg_cv_auc > best_auc:
                best_auc = avg_cv_auc
                best_params = {'rank': r, 'lambda': l, 'alpha': a}

print(f"\n--- GRID SEARCH COMPLETE ---")
print(f"Best Parameters: {best_params}")
print(f"Best CV AUC: {best_auc:.4f}")

Splitting training data into 5 folds...
Starting Scalable Triple-Loop Grid Search...


## 8. Champion Model Training & Final Metrics Validation
With the optimal hyperparameters identified, we train our final "Champion" ALS model on the entire 80% training split. To strictly comply with the assignment, the predictions and actual labels are flattened and fed into Spark's native `BinaryClassificationMetrics` API to calculate the final test AUC. We also utilize a distributed Map-Reduce job to calculate the confusion matrix, yielding our final Top-50 Precision, Recall, and Accuracy.

In [ ]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics

print("Training Final Champion Model...")
final_model = ALS.trainImplicit(
    trainData_final, 
    rank=best_params.get('rank', 50),  # Dynamically use best params
    iterations=10, 
    lambda_=best_params.get('lambda', 0.01), 
    alpha=best_params.get('alpha', 1.0)
)

print("Evaluating Final Model on Test Data...")

# 1. Get total unique artists for the True Negative calculation
total_artists_count = trainData_final.map(lambda r: r.product).distinct().count()

# 2. Group the test data by user
test_user_artists = testData_final.map(lambda r: (r.user, r.product)).groupByKey().mapValues(set)

# 3. Generate Top-50 recommendations for ALL users in ONE distributed job
final_recs = final_model.recommendProductsForUsers(50)

# 4. Join predictions with actual ground truth
eval_rdd = final_recs.join(test_user_artists)

# 5. Build global predictions and labels RDD to comply strictly with the assignment
def build_metrics_pairs(row):
    user, (recs, actual_artists) = row
    return [(float(r.rating), 1.0 if r.product in actual_artists else 0.0) for r in recs]

# Flatten all user pairs into a single RDD for the API
global_predictions_and_labels = eval_rdd.flatMap(build_metrics_pairs)

# 6. Execute the required Spark API
metrics = BinaryClassificationMetrics(global_predictions_and_labels)
final_auc = metrics.areaUnderROC

# 7. Distributed Map-Reduce for Precision, Recall, and Accuracy
def compute_confusion_matrix(row):
    user, (recs, actual_artists) = row
    rec_products = set([r.product for r in recs])
    
    hits = len(rec_products.intersection(actual_artists))
    tp = hits
    fp = 50 - hits
    fn = len(actual_artists) - hits
    tn = total_artists_count - 50 - len(actual_artists) + hits
    
    return (tp, fp, fn, tn)

# Reduce across all worker nodes
cm_totals = eval_rdd.map(compute_confusion_matrix).reduce(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2], a[3]+b[3]))

final_tp, final_fp, final_fn, final_tn = cm_totals

precision = final_tp / (final_tp + final_fp) if (final_tp + final_fp) > 0 else 0
recall = final_tp / (final_tp + final_fn) if (final_tp + final_fn) > 0 else 0
accuracy = (final_tp + final_tn) / (final_tp + final_fp + final_tn + final_fn) if (final_tp + final_fp + final_tn + final_fn) > 0 else 0

print(f"--- FINAL MODEL METRICS ---")
print(f"Test AUC:      {final_auc:.4f}")
print(f"Test Precision (Top 50): {precision:.4f}")
print(f"Test Recall    (Top 50): {recall:.4f}")
print(f"Test Accuracy  (Top 50): {accuracy:.4f}")

Training Final Champion Model...


26/05/01 22:02:20 WARN BlockManager: Task 5323918 already completed, not releasing lock for rdd_69_0


Evaluating Final Model on Test Data...


--- FINAL MODEL METRICS ---
Test AUC:      0.5870
Test Precision (Top 50): 0.1178
Test Recall    (Top 50): 0.0890
Test Accuracy  (Top 50): 0.0534


## 9. Dictionary Lookup for Personalization
Before we can test the recommender system on a custom profile, we need to map the numeric IDs back to human-readable strings. I parse the `artist_data.txt` file and run a quick filter to find the exact IDs for some of my personal favorite bands to use in the next step.

In [ ]:
# Load the raw artist names to find your favorites
raw_artist_data = sc.textFile(f"{PATH_TO_DATA}/artist_data.txt")

def parse_artist(line):
    try:
        tokens = line.split('\t')
        return (int(tokens[0]), tokens[1].strip())
    except:
        return None

artist_names = raw_artist_data.map(parse_artist).filter(lambda x: x is not None)
artist_names_map = artist_names.collectAsMap()


search_terms = ["daft punk", "metallica", "adele", "linking park", "eminem"] 

for term in search_terms:
    print(f"\nSearching for: {term}")
    results = artist_names.filter(lambda x: term.lower() in x[1].lower()).take(3)
    for res in results:
        print(f"ID: {res[0]} | Name: {res[1]}")


Searching for: daft punk
ID: 1240304 | Name: DJ Sneak & Daft Punk
ID: 1135418 | Name: Daft Punk with The Chemical Brothers
ID: 1244878 | Name: Super 2000 vs. Daft Punk

Searching for: metallica
ID: 10584626 | Name: Kool G Rap Vs. Metallica
ID: 10584827 | Name: Metallica & Queens Of The Stone Age
ID: 1240322 | Name: Metallica & The SF Symphony Orchestra

Searching for: adele
ID: 10025079 | Name: Fred & Adele Astaire
ID: 10028670 | Name: Alexandre Tharaud - piano/Madeleine Milhaud - narrator
ID: 10476797 | Name: Donovan/Jeff Beck Group, Lesley and Madeleine

Searching for: linking park


ID: 10620436 | Name: Linking Park and Jay-Z
ID: 10041400 | Name: Jay-Z & Linking Park
ID: 9963115 | Name: Jay Z vs Linking Park

Searching for: eminem
ID: 6918062 | Name: Dr.Dre, Eminem, Nate Dogg, Snoop Dogg, Xzibit
ID: 1135032 | Name: Dr. Dre feat. Xzibit Eminem
ID: 10435135 | Name: Lloyd Banks Feat. 50 Cent,Eminem & Nate Dogg


## 10. Custom Profile Injection & Qualitative Evaluation (Step C)
To truly test if the matrix factorization learned meaningful latent features (genres, eras, vibes), I create a brand new user profile (`User 9999999`) seeded with my personal ratings. We inject this into the training data and retrain the champion model one last time. By generating the Top 25 recommendations for this custom ID, we can qualitatively judge the model's performance against actual human music taste.

In [ ]:
# 1. Define your personal profile
MY_USER_ID = 9999999

# Add your chosen Artist IDs here. 
my_ratings = [
    Rating(MY_USER_ID, 1240304, 100), # Example ID
    Rating(MY_USER_ID, 1240322, 66),  # Example ID
    Rating(MY_USER_ID, 10041400, 80),  # Example ID
    Rating(MY_USER_ID, 6918062, 70),  # Example ID
    Rating(MY_USER_ID, 10476797, 90),  # Example ID
]

# 2. Inject your profile into the training data
my_ratings_rdd = sc.parallelize(my_ratings)
personalized_train_data = trainData_final.union(my_ratings_rdd)

# 3. Train the model using your Champion hyper-parameters
print("Training personalized model...")
personalized_model = ALS.trainImplicit(
    personalized_train_data, 
    rank=50, 
    iterations=10, 
    lambda_=0.01, 
    alpha=1.0
)

# 4. Generate Top 25 Recommendations for YOU
print("\n--- YOUR TOP 25 RECOMMENDATIONS ---")
my_recs = personalized_model.call("recommendProducts", MY_USER_ID, 25)

for i, rec in enumerate(my_recs, 1):
    # Map the recommended ID back to the artist's real name
    artist_name = artist_names_map.get(rec.product, "Unknown Artist")
    print(f"{i}. {artist_name} (Score: {rec.rating:.4f})")